# Multicell UQ and sensitivity analysis (WEPB015)

This example reproduces the workflow of

> S. Udongwo and U. van Rienen, *Multicell parameterisation for sensitivity analysis and uncertainty quantification of accelerator cavities*, **IPAC 2025**, WEPB015. [doi:10.18429/JACoW-IPAC2025-WEPB015](https://doi.org/10.18429/JACoW-IPAC2025-WEPB015)

Real cavities are welded from separately-manufactured **dumb-bells** (half-cells), so manufacturing tolerances perturb each half-cell independently. The **multicell parameterisation** gives every half-cell its own variables; continuity at the welded irises and equators is then restored by *averaging* the joined half-cells. This example:

1. perturbs the iris (`Ri`) and equator (`Req`) radii of every half-cell independently and welds them back (Monte-Carlo),
2. plots the figure-of-merit **distributions** (Fig. 3) and the before/after-continuity node spread (Fig. 4),
3. summarises each FM as a **mean ± standard deviation** with the nominal design point (Fig. 5),
4. ranks the inputs by **Sobol' sensitivity indices** (Fig. 6), and
5. overlays the accelerating **TM$_{010}$ passband** on the European XFEL measurements of Corno *et al.*

> It runs the paper's **9-cell TESLA** cavity (so the TM$_{010}$ passband matches the European XFEL data), but with a **small Monte-Carlo sample count** so the docs build in reasonable time. Raise `N_SAMPLES` to ~900 to reproduce the paper's numbers (much slower). At this small sample count some figure-of-merit surrogates are under-resolved — the *surrogate-quality* check below makes that explicit.

In [ ]:
import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

from cavsim2d import Study, EllipticalCavity
from cavsim2d.utils.style import apply_style

apply_style()
N_SAMPLES = 500   # Monte-Carlo geometries (small for a quick docs build; ~900 for the paper)

## 1. The cavity and the uncertainty

A TESLA-like cell in a one-cavity study (UQ perturbs the geometry, so it runs through `Study`). The `uq_config` uses the **multicell** complexity with `independent_half_cells=True`: each half-cell's `Ri`/`Req` is drawn from a Gaussian (σ = 0.3 mm, `method=['normal', N]`), then the shared seams are welded (averaged). Setting the objectives to the figures of merit records them for every perturbed geometry in `uq/table.csv`.

In [2]:
cavs = Study(r'C:\Users\Soske\Documents\git_projects\cavsim2d_simulations\testdelete', 'multicell_uq')
midcell = [42, 42, 12, 19, 35, 57.652, 103.3536]  # <- A, B, a, b, Ri, L, Req
endcell_l = [40.34, 40.34, 10, 13.5, 39, 55.7251, 103.3536]
endcell_r = [42, 42, 9, 12.8, 39, 56.8407, 103.3536]
cav = EllipticalCavity(9, midcell, endcell_l, endcell_r, beampipe='both', name='tesla')
cavs.add_cavity([cav], ['TESLA'])

tune_config = {
        'freqs': 1300,
        # 'cell_type': {'end-cell': 'L'},
    }
cavs.run_eigenmode({
    'n_cells': 9, 'processes': 1, 'boundary_conditions': 'mm',
    'uq_config': {
        'cell_complexity': 'multicell',
        'independent_half_cells': True,
        'variables': ['Ri', 'Req'],
        'method': ['normal', N_SAMPLES],
        'perturbation_mode': ['add', 0.3],   # sigma = 0.3 mm
        'delta': [0.3, 0.3],
        'objectives': ['monopole:freq [MHz]', 'monopole:Epk/Eacc []',
                       'monopole:Bpk/Eacc [mT/MV/m]', 'monopole:R/Q [Ohm]',
                       'monopole:G [Ohm]'],
        'processes': 12,
        'tune_config': tune_config
    },
})

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

## 2. Figure-of-merit distributions (paper Fig. 3)

Each perturbed geometry contributes one row to `uq/table.csv`; the histogram of every figure of merit shows how the tolerance band spreads the RF performance.

In [ ]:
cav.eigenmode.plot_fm_distribution()

## 3. Before vs after welding (paper Fig. 4)

Each half-cell's radius is perturbed independently (**before** continuity); welding averages the two half-cells that meet at a shared equator, which **narrows** that distribution. The apertures (`Ri`) of a single cell are not shared, so their before/after distributions coincide; the equator (`Req`) is shared and visibly tightens.

In [ ]:
cav.eigenmode.plot_node_distribution(['Ri', 'Req'])

## 4. Mean and standard deviation (paper Fig. 5)

The built-in UQ figure-of-merit plot: each QOI as a mean with a ±1σ error bar and the nominal design point.

In [ ]:
cav.eigenmode.plot_fm_scatter(uq=True)

## 5. Sobol' sensitivity indices (paper Fig. 6)

`run_sensitivity` fits a low-order polynomial surrogate to the samples (`uq/nodes.csv` -> `uq/table.csv`) and evaluates Sobol' **main** ($S_1$) and **total** ($S_T$) indices over a Saltelli quasi-Monte-Carlo design — the paper's PCE + QMC method, made cheap. The frequency is dominated by the equator radius; the peak-field ratios and $R/Q$ by the iris radii. A wide $S_1$–$S_T$ gap flags higher-order interactions.

In [ ]:
sobol = cav.eigenmode.run_sensitivity(N=256)
cav.eigenmode.plot_sobol_indices()

### How good is the surrogate?

The Sobol' indices are only as trustworthy as the polynomial surrogate they are read from. `surrogate_quality()` reports the fit per figure of merit and `plot_surrogate_quality()` draws the **cross-validated** prediction against the true value — points on the diagonal mean a faithful surrogate. A low $R^2$, or a large gap between $R^2$ and the cross-validated $R^2$, means that FM needs more samples or a higher polynomial degree before its indices can be trusted (here `freq` and `R/Q` fit almost perfectly; a peak-field ratio may not, at this small sample count).

In [ ]:
cav.eigenmode.surrogate_quality().round(5)

In [ ]:
cav.eigenmode.plot_surrogate_quality()

In [ ]:
import os
d = cav.eigenmode.cavity.uq_dir
for f in ('nodes.csv','table.csv','surrogate.json','sobol.json'):
    print(f, os.path.exists(os.path.join(d, f)))


## 6. The TM$_{010}$ accelerating passband

The fundamental monopole passband has one mode per cell. For a 9-cell cavity its nine frequencies can be compared with the European XFEL measurements of

> J. Corno, N. Georg, S. Gorgi Zadeh *et al.*, *Uncertainty modeling and analysis of the European X-ray Free Electron Laser cavities manufacturing process*, **Nucl. Instrum. Methods A 971** (2020) 164135. [doi:10.1016/j.nima.2020.164135](https://doi.org/10.1016/j.nima.2020.164135)

Their Table 1 gives the averaged TM$_{010}$ spectra for two vendors (RI and EZ). `plot_tm010_spectrum` overlays both as **probability-of-resonance distributions** (each mode a Gaussian of its manufacturing spread — the Corno Fig. 5 view) with this cavity's deterministic modes as vertical lines. Pass `kde=False` instead for a frequency-vs-mode-index plot with error bars.

In [ ]:
# Corno et al. (2020), Table 1 — averaged TM010 spectra [MHz], two vendors (RI, EZ)
corno = {
    'DESY RI': {'mean': [1275.832, 1277.951, 1281.194, 1285.178, 1289.423,
                         1293.442, 1296.762, 1298.944, 1299.702],
                'std':  [0.226, 0.212, 0.186, 0.157, 0.124, 0.089, 0.064, 0.052, 0.050]},
    'DESY EZ': {'mean': [1274.734, 1276.926, 1280.294, 1284.460, 1288.921,
                         1293.161, 1296.649, 1298.946, 1299.716],
                'std':  [0.268, 0.245, 0.208, 0.168, 0.129, 0.090, 0.066, 0.054, 0.052]},
}
# The passband comes straight from the UQ study's cavity (its nominal solve
# already computed the monopole modes) — no separate run needed.
cav.eigenmode.plot_tm010_spectrum(reference=corno, uq=True, density_cap=10.5)

In [ ]:
cav.eigenmode.plot_spectra('freq [MHz]', density_cap=5.2, break_axis=True, freq_range=[[1275, 1301], [2360, 2460]])
import matplotlib.pyplot as plt
%matplotlib widget
plt.show()

See also the [eigenmode UQ example](../advanced/eigenmode_uq.ipynb) (figure-of-merit spread across cavity types) and the [eigenmode spectrum](elliptical_tesla.ipynb) view.